# 00 Setup and acquisition

Pulls both datasets into `data/raw`, loads them, and saves them unmodified as Parquet for the next stage.

| Dataset | Kaggle ref | Version | Pulled |
|---|---|---|---|
| Steam | `fronkongames/steam-games-dataset` | v49 | 19 Sep 2026 |
| HowLongToBeat | `b4n4n4p0wer/how-long-to-beat-video-game-playtime-dataset` | v5 | 19 Sep 2026 |

The Steam dataset refreshes continuously. Do not re-download mid-project.

**Reads:** `data/raw/steam_v49.zip`, `data/raw/hltb_v5.zip`  
**Writes:** `data/interim/00_steam.parquet`, `data/interim/00_hltb.parquet`

In [ ]:
import sys
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists()) / "src"))
from is630 import config

In [ ]:
# Download only if the pinned archive is missing
for path, ref in {config.STEAM_ZIP: config.STEAM_REF, config.HLTB_ZIP: config.HLTB_REF}.items():
    if not path.exists():
        urllib.request.urlretrieve(f"https://www.kaggle.com/api/v1/datasets/download/{ref}", path)
    print(path.name, f"{path.stat().st_size / 1e6:.1f} MB")

## Steam

The CSV header fuses `Discount` and `DLC count` into one name, giving 39 headers for 40 fields. Reading it as-is shifts every column one place, so the header is split before loading.

In [ ]:
with zipfile.ZipFile(config.STEAM_ZIP) as z:
    with z.open("games.csv") as f:
        header = pd.read_csv(f, nrows=0).columns.tolist()

i = header.index("DiscountDLC count")
names = header[:i] + ["Discount", "DLC count"] + header[i + 1:]

with zipfile.ZipFile(config.STEAM_ZIP) as z:
    with z.open("games.csv") as f:
        steam = pd.read_csv(f, header=0, names=names, index_col=False)

steam.shape

In [ ]:
steam.head()

## HowLongToBeat

In [ ]:
with zipfile.ZipFile(config.HLTB_ZIP) as z:
    with z.open("hltb_dataset_normalized.csv") as f:
        hltb = pd.read_csv(f)

hltb.shape

In [ ]:
hltb.head()

## Save for stage 01

Both tables are saved as loaded. Column pruning and type cleaning belong to later stages.

In [ ]:
steam.to_parquet(config.INTERIM / "00_steam.parquet", index=False)
hltb.to_parquet(config.INTERIM / "00_hltb.parquet", index=False)

for name, df in [("00_steam", steam), ("00_hltb", hltb)]:
    size = (config.INTERIM / f"{name}.parquet").stat().st_size / 1e6
    print(f"{name}: {len(df):,} rows out, {size:.1f} MB")